# LFM3-Audio-2B — ASR, TTS, and audio understanding on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/LFM3-Audio-2B-voice.ipynb)

[`LiquidAI/LFM3-Audio-2B`](https://huggingface.co/LiquidAI/LFM3-Audio-2B) is Liquid AI's **audio** LFM3 (~2.00B, `lfm3_audio`). Text or 16 kHz audio in; text, 24 kHz speech, or interleaved text+speech out. It ships an in-house causal audio encoder and **LiquidCodec** tokenizer (`audio_tokenizer/` on the Hub).

Card install: `transformers>=5.20.2` plus `librosa` and `soundfile`, with `trust_remote_code`. Native support is still landing in [`huggingface/transformers#50036`](https://github.com/huggingface/transformers/pull/50036).

**Licence:** [LFM Open License v1.0](https://huggingface.co/LiquidAI/LFM3-Audio-2B/blob/main/LICENSE) (`license: other`, `lfm1.0`).

**Runtime:** *Runtime > Change runtime type > T4 GPU*. Load **float16** + **SDPA**. About 4.0 GB of backbone weights plus the codec. Keep clips short (a few seconds) and `max_new_tokens` small. TTS uses the card's audio sampler (`audio_temperature=1.0`, `audio_top_k=128`, `audio_repetition_penalty=1.1`).

**Links:** [model card](https://huggingface.co/LiquidAI/LFM3-Audio-2B) · [blog](https://www.liquid.ai/blog/lfm3) · [announcement](https://x.com/liquidai/status/2115128329003303107) · [collection](https://huggingface.co/collections/LiquidAI/lfm3) · [GGUF](https://huggingface.co/LiquidAI/LFM3-Audio-2B-GGUF)


In [ ]:
%pip install -q -U "transformers>=5.20.2" accelerate
%pip install -q librosa soundfile

# Card pin: transformers>=5.20.2, torch, accelerate, librosa, soundfile.
# Do not reinstall torch. The codec implementation is in the Hub repo;
# the processor loads weights from audio_tokenizer/.


> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.


## 1. Check the T4 and pick a dtype


In [ ]:
import os
import torch
import transformers
from packaging.version import Version

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

assert Version(transformers.__version__) >= Version("5.20.2"), (
    f"transformers {transformers.__version__} is loaded; this notebook needs 5.20.2+. "
    "Run the install cell, then Runtime > Restart session."
)

if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

# T4 (sm_75) has no FlashAttention-2. Keep PyTorch SDPA on mem-efficient and math kernels.
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

# The cards default to bfloat16 on GPU. A T4 has no native BF16, so use float16.
major, minor = torch.cuda.get_device_capability()
DTYPE = torch.float16
DEVICE = "cuda"
props = torch.cuda.get_device_properties(0)
print("transformers", transformers.__version__)
print("torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(), f"sm_{major}{minor}", "| dtype:", DTYPE)
print(f"VRAM total: {props.total_memory / 1024**3:.1f} GB")
print("FlashAttention-2: off (sdpa / eager only)")
if major < 8:
    print("No native bfloat16 on this GPU; loading float16.")
if major >= 8:
    print("This GPU can run bfloat16; this notebook still uses float16 so a T4 recipe stays consistent.")


## 2. Load LFM3-Audio-2B

`AutoModelForMultimodalLM` + `AutoProcessor`, then move `processor.audio_tokenizer` to the GPU, as on the card. Text-only generate sets `suppress_mixed=True` and `suppress_audio_out=True`.


In [ ]:
from transformers import AutoModelForMultimodalLM, AutoProcessor

MODEL_ID = "LiquidAI/LFM3-Audio-2B"

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()


def load_audio(attn):
    return AutoModelForMultimodalLM.from_pretrained(
        MODEL_ID,
        dtype=DTYPE,
        device_map=DEVICE,
        attn_implementation=attn,
        trust_remote_code=True,
    ).eval()


ATTN = None
model = None
last_error = None
for attn in ("sdpa", "eager"):
    try:
        model = load_audio(attn)
        ATTN = attn
        break
    except Exception as exc:
        last_error = exc
        print(f"{attn} float16 load failed:", exc)
        del model
        model = None
        torch.cuda.empty_cache()

if model is None:
    raise RuntimeError("could not load LFM3-Audio-2B") from last_error

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
codec = processor.audio_tokenizer.to(device=DEVICE).eval()

print("attention:", ATTN, "| dtype:", getattr(model, "dtype", DTYPE))
print(f"parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")
print(
    f"VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB | "
    f"peak: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB"
)


## 3. Public speech fixture

A short public LibriSpeech dummy clip (not gated). Fallback: synthesize a sentence with `espeak-ng` if the download fails.


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

import librosa
import numpy as np
import soundfile as sf
from IPython.display import Audio, display

ASR_URL = "https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/1.flac"
ASR_PATH = Path("lfm3_asr_dummy.wav")


def download_or_synth():
    try:
        src = Path("lfm3_asr_dummy.flac")
        urlretrieve(ASR_URL, src)
        wav, sr = librosa.load(src, sr=16_000, mono=True)
        sf.write(ASR_PATH, wav, 16_000)
        print(f"downloaded {ASR_URL} -> {ASR_PATH} ({len(wav)/16000:.1f}s)")
        return ASR_PATH
    except Exception as exc:
        print("download failed, synthesizing with espeak-ng:", exc)
        import shutil
        import subprocess

        if not shutil.which("espeak-ng"):
            subprocess.run(["apt-get", "-qq", "update"], check=True)
            subprocess.run(["apt-get", "-qq", "install", "-y", "espeak-ng"], check=True)
        subprocess.run(
            ["espeak-ng", "-s", "140", "-v", "en-us", "-w", str(ASR_PATH),
             "The birch canoe slid on the smooth planks."],
            check=True,
        )
        return ASR_PATH


audio_path = download_or_synth()
wav, sr = sf.read(audio_path)
display(Audio(wav, rate=sr))


## 4. Demo 1 — speech recognition (ASR)

System prompt from the card: `Perform ASR in English.`


In [ ]:
conversation = [
    {"role": "system", "content": "Perform ASR in English."},
    {"role": "user", "content": [{"type": "audio", "audio": str(audio_path)}]},
]
inputs = processor.apply_chat_template(
    conversation,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(DEVICE)
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        suppress_mixed=True,
        suppress_audio_out=True,
        return_dict_in_generate=True,
    )
text = processor.decode(
    outputs.sequences[0, inputs["input_ids"].shape[1] :],
    skip_special_tokens=True,
)
print(text)
print(
    f"VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB | "
    f"peak: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB"
)


## 5. Demo 2 — short TTS

Card prompt: `Perform TTS in American English using a female voice.` Prefill `<|audio_start|>`. Output is 24 kHz.


In [ ]:
conversation = [
    {"role": "system", "content": "Perform TTS in American English using a female voice."},
    {"role": "user", "content": "The quick brown fox jumps over the lazy dog."},
    {"role": "assistant", "content": "<|audio_start|>"},
]
inputs = processor.apply_chat_template(
    conversation,
    continue_final_message=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(DEVICE)
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        audio_temperature=1.0,
        audio_top_k=128,
        audio_repetition_penalty=1.1,
        audio_repetition_window=4096,
        return_dict_in_generate=True,
    )
    waveform = codec.decode(
        outputs.audio_codes[:1, :, : outputs.audio_lengths[0]]
    ).audio_values[0]

tts_path = Path("lfm3_audio_tts.wav")
audio_np = waveform.float().detach().cpu().numpy()
if audio_np.ndim > 1:
    audio_np = np.squeeze(audio_np)
sf.write(tts_path, audio_np, 24_000)
print(f"wrote {tts_path} ({len(audio_np) / 24000:.1f}s @ 24 kHz)")
display(Audio(str(tts_path)))
print(
    f"VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB | "
    f"peak: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB"
)


## 6. Demo 3 — audio understanding

Same clip, asked as a grounded question. Text-only output (`suppress_mixed` / `suppress_audio_out`).


In [ ]:
conversation = [
    {
        "role": "system",
        "content": (
            "Answer questions about the provided audio clip. "
            "Base every answer only on what can be heard."
        ),
    },
    {
        "role": "user",
        "content": [
            {"type": "audio", "audio": str(audio_path)},
            {"type": "text", "text": "Is someone speaking, and in what language?"},
        ],
    },
]
inputs = processor.apply_chat_template(
    conversation,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(DEVICE)
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        suppress_mixed=True,
        suppress_audio_out=True,
        return_dict_in_generate=True,
    )
print(processor.decode(
    outputs.sequences[0, inputs["input_ids"].shape[1] :],
    skip_special_tokens=True,
))
print(
    f"VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB | "
    f"peak: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB"
)


## Optional GGUF

[`LiquidAI/LFM3-Audio-2B-GGUF`](https://huggingface.co/LiquidAI/LFM3-Audio-2B-GGUF) includes `Q4_K_M` (~1.1 GB) and an mmproj. Multimodal llama.cpp for LiquidCodec is more moving than the text GGUF; this notebook stays on Transformers.


## Caveats

- **Licence:** [LFM Open License v1.0](https://huggingface.co/LiquidAI/LFM3-Audio-2B/blob/main/LICENSE) (`license: other`).
- **T4 caps:** short clips, `max_new_tokens` 128–256, float16. Interleaved voice-chat is heavier; this notebook uses text-only ASR/understanding plus one short TTS.
- **Install:** `transformers>=5.20.2` + `trust_remote_code`. Native LFM3-Audio is still [PR #50036](https://github.com/huggingface/transformers/pull/50036).
- **HF token:** public weights. Cancel Colab `HF_TOKEN` prompts.
- **Not executed here:** no GPU in the agent VM. Full T4 `generate()` was not run.
- **Do not merge from this agent.**
